# Modeling Lab (ARIMA + GRU)

This notebook trains simple forecasting models **locally** from the Postgres data in `raw.eth_ohlcv` (1h candles) and saves model artifacts into `ml/model_registry/`.

Models:
- **ARIMA** (statsmodels): fast CPU baseline
- **GRU** (PyTorch): lightweight deep learning multi-step forecast (next 6 hours)

After training, you can rebuild the Docker `api`/`web` containers so the Web UI can fetch forecasts via the API.


In [1]:
import sys
print("python:", sys.executable)
print("version:", sys.version)


python: /Library/Developer/CommandLineTools/usr/bin/python3
version: 3.9.6 (default, Apr 30 2025, 02:07:17) 
[Clang 17.0.0 (clang-1700.0.13.5)]


## Optional: install deps for this kernel

If imports fail below, uncomment and run.


In [2]:
# If needed:
!"{sys.executable}" -m pip install -q pandas psycopg2-binary statsmodels
!"{sys.executable}" -m pip install -q torch


You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [3]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import psycopg2


def load_env_file(path: str = ".env") -> dict:
    p = Path(path)
    if not p.exists():
        return {}
    out = {}
    for line in p.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        k, v = line.split("=", 1)
        out[k.strip()] = v.strip().strip("\"'")
    return out


env = {**load_env_file(".env"), **os.environ}
db_host = env.get("DB_HOST", "localhost")
if db_host == "postgres":
    # docker-internal hostname; from your Mac/Jupyter you want localhost
    db_host = "localhost"

conn = psycopg2.connect(
    host=db_host,
    port=int(env.get("DB_PORT", "5432")),
    user=env.get("DB_USER"),
    password=env.get("DB_PASSWORD"),
    dbname=env.get("DB_NAME"),
)
print("connected:", db_host, env.get("DB_NAME"), "as", env.get("DB_USER"))


OperationalError: connection to server at "localhost" (::1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?
connection to server at "localhost" (127.0.0.1), port 5432 failed: Connection refused
	Is the server running on that host and accepting TCP/IP connections?


In [ ]:
# Load raw 1h candles
EXCHANGE = env.get("EXCHANGE", "binance")
SYMBOL = env.get("SYMBOL", "ETH/USDT")

df = pd.read_sql(
    """
    SELECT ts, open, high, low, close, volume
    FROM raw.eth_ohlcv
    WHERE exchange = %s AND symbol = %s AND timeframe = '1h'
    ORDER BY ts
    """,
    conn,
    params=(EXCHANGE, SYMBOL),
)

print("rows:", len(df))
df.head()


In [ ]:
# Basic cleaning / diagnostics
if df.empty:
    raise SystemExit("No rows found in raw.eth_ohlcv")

df = df.dropna(subset=["ts", "close"]).copy()
df["ts"] = pd.to_datetime(df["ts"], utc=True)
df = df.sort_values("ts")
df = df.drop_duplicates(subset=["ts"], keep="last")
df = df.set_index("ts")

deltas = df.index.to_series().diff().dropna()
expected = pd.Timedelta(hours=1)
gap_count = int((deltas != expected).sum())
print("start:", df.index.min())
print("end:", df.index.max())
print("gaps:", gap_count)

# Optional: fill missing hours (choose one)
FILL_MISSING = False
if FILL_MISSING:
    full_index = pd.date_range(df.index.min(), df.index.max(), freq="H", tz="UTC")
    df = df.reindex(full_index)
    # Simple fill: forward-fill OHLC, 0 volume; then drop any leading NaNs
    df[["open", "high", "low", "close"]] = df[["open", "high", "low", "close"]].ffill()
    df["volume"] = df["volume"].fillna(0.0)
    df = df.dropna(subset=["close"])
    print("after fill rows:", len(df))

df.tail()


## ARIMA (fast baseline)

This trains ARIMA on the `close` series and saves it to `ml/model_registry/eth_arima_h6.pkl`.


In [ ]:
HORIZON = 6
ORDER = (2, 1, 2)  # (p,d,q)

from statsmodels.tsa.arima.model import ARIMA

series = df["close"].astype(float).to_numpy()
model = ARIMA(series, order=ORDER)
results = model.fit()

pred = results.forecast(steps=HORIZON)
future_index = pd.date_range(df.index.max() + pd.Timedelta(hours=1), periods=HORIZON, freq="H", tz="UTC")
arima_forecast = pd.DataFrame({"ts": future_index, "pred_close": np.asarray(pred, dtype=float)})

out_path = Path("ml/model_registry/eth_arima_h6.pkl")
out_path.parent.mkdir(parents=True, exist_ok=True)
results.save(str(out_path))
print("saved:", out_path)
arima_forecast


## GRU (deep learning)

This calls the existing training script `ml/train_deep_forecast.py` and saves `ml/model_registry/eth_gru_h6.pt`.

If this cell fails on `torch` install, it's almost always because your kernel is using a Python that doesn't have a compatible torch wheel.


In [ ]:
# Train GRU (edit epochs/lookback if you want)
!"{sys.executable}" -m ml.train_deep_forecast --lookback 168 --horizon 6 --epochs 20 --device auto --model_out ml/model_registry/eth_gru_h6.pt


In [ ]:
# Predict next 6 hours with the saved GRU model
!"{sys.executable}" -m ml.predict_deep_forecast --model ml/model_registry/eth_gru_h6.pt


## Make It Show In Web UI

The API container copies `ml/model_registry/` into `/app/model_registry` at build time. After training locally, rebuild:

```bash
cd /Users/ch/Desktop/crypto_predict
docker compose up -d --build api web
```

Then:
- API model list: `http://localhost:8000/predict/models`
- Forecast: `http://localhost:8000/predict/forecast?model=arima` or `model=gru`
- UI: `http://localhost:3000`


In [ ]:
conn.close()
print("closed")
